In [1]:

import pandas as pd
import numpy as np
import os
import joblib

PROCESSED_DIR = "../data/processed/"
MODELS_DIR    = "../models/"

# Load the saved pipeline (preprocessor + model together)
model = joblib.load(MODELS_DIR + "rf_final.joblib")

# Load the cleaned test data
test_df = pd.read_csv(PROCESSED_DIR + "test_clean.csv")

# Fix missing Fare (safety net — same as Day 5)
train_df = pd.read_csv(PROCESSED_DIR + "train_clean.csv")
test_df["Fare"] = test_df["Fare"].fillna(train_df["Fare"].median())

print("Model loaded:", type(model).__name__)
print("Test shape  :", test_df.shape)
print("Test columns:", test_df.columns.tolist())
print("Missing     :", test_df.isnull().sum().sum())

Model loaded: Pipeline
Test shape  : (418, 11)
Test columns: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone', 'HasCabin']
Missing     : 0


In [2]:

predictions = model.predict(test_df)
probabilities = model.predict_proba(test_df)

print("Predictions shape:", predictions.shape)
print("Probabilities shape:", probabilities.shape)

print("\n=== Prediction Distribution ===")
print(pd.Series(predictions).value_counts())

print("\n=== First 10 Predictions ===")
print("Predicted class:", predictions[:10])
print("Predicted proba:")
print(probabilities[:10].round(4))

Predictions shape: (418,)
Probabilities shape: (418, 2)

=== Prediction Distribution ===
0    254
1    164
Name: count, dtype: int64

=== First 10 Predictions ===
Predicted class: [0 1 0 0 1 0 1 0 1 0]
Predicted proba:
[[0.902  0.098 ]
 [0.4395 0.5605]
 [0.8857 0.1143]
 [0.8696 0.1304]
 [0.3906 0.6094]
 [0.8775 0.1225]
 [0.3754 0.6246]
 [0.8706 0.1294]
 [0.2843 0.7157]
 [0.9104 0.0896]]


In [4]:

import os


raw_test = pd.read_csv("../data/raw/test.csv")

submission = pd.DataFrame({
    "PassengerId": raw_test["PassengerId"],
    "Survived":    predictions
})

# Save to project root
SUBMISSION_PATH = "../submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

print("Saved:", SUBMISSION_PATH)
print("Shape:", submission.shape)
print("\nFirst 10 rows:")
print(submission.head(10))
print("\nPrediction distribution:")
print(submission["Survived"].value_counts())

Saved: ../submission.csv
Shape: (418, 2)

First 10 rows:
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1
5          897         0
6          898         1
7          899         0
8          900         1
9          901         0

Prediction distribution:
Survived
0    254
1    164
Name: count, dtype: int64


In [5]:

import os

SUBMISSION_PATH = "../submission.csv"

print("=== Final Sanity Checks ===")

# 1. File exists
print(f"\n1. File exists: {os.path.exists(SUBMISSION_PATH)}")

# 2. Read it back
sub = pd.read_csv(SUBMISSION_PATH)
print(f"\n2. Reloaded shape: {sub.shape}")
print(f"   Expected       : (418, 2)")

# 3. Correct columns
print(f"\n3. Columns: {sub.columns.tolist()}")
print(f"   Expected: ['PassengerId', 'Survived']")

# 4. No missing
print(f"\n4. Missing values: {sub.isnull().sum().sum()}")

# 5. Only 0/1 in Survived
print(f"\n5. Survived unique values: {sorted(sub['Survived'].unique())}")

# 6. PassengerId range
print(f"\n6. PassengerId range: {sub['PassengerId'].min()} to {sub['PassengerId'].max()}")
print(f"   Expected: 892 to 1309")

# 7. Row count matches test
print(f"\n7. Rows: {len(sub)} (should be 418)")

=== Final Sanity Checks ===

1. File exists: True

2. Reloaded shape: (418, 2)
   Expected       : (418, 2)

3. Columns: ['PassengerId', 'Survived']
   Expected: ['PassengerId', 'Survived']

4. Missing values: 0

5. Survived unique values: [np.int64(0), np.int64(1)]

6. PassengerId range: 892 to 1309
   Expected: 892 to 1309

7. Rows: 418 (should be 418)
